# Regularized Matrix Factorization

This collaborative filtering technique predicts user preferences by decomposing a spare user-item interaction matrix into lower-dimenssional, dense user and item latent factor matrices. It adds regularization terms to the loss function to penalize large factor values, preventing overfitting and improving generalization on unseen data.

### 1. Load Dataset

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from math import sqrt


In [3]:
df = pd.read_csv("../data/ratings_small.csv")

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100004 entries, 0 to 100003
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100004 non-null  int64  
 1   movieId    100004 non-null  int64  
 2   rating     100004 non-null  float64
 3   timestamp  100004 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


### 2. Clean data, split into train/test (based on timestamp)

We want to drop sparse users, then base the train/test split on the timestamp a user has watched the movie. 

In [4]:
# drop users with less than 5 ratings
counts = df["userId"].value_counts()
df = df[df['userId'].isin(counts[counts >= 5].index)].copy()

# split into train and test sets -- 80% train, 20% test
df.sort_values(['userId', 'timestamp'], inplace=True)

train, test = [], []
for userID, group in df.groupby('userId'):
    cut = int(len(group) * 0.8)
    train.append(group.iloc[:cut])
    test.append(group.iloc[cut:])
train_set = pd.concat(train).reset_index(drop=True)
test_set = pd.concat(test).reset_index(drop=True)

print(f"\nTrain: {len(train_set):,} | Test: {len(test_set):,}")



Train: 79,748 | Test: 20,256


### 3. Index encoding

Since the movie/user IDs are arbitrary numbers, we want to map them to contiguous integer indices after filtering through each entry

In [5]:
#get all unique users and movies
all_users = df['userId'].unique()
all_movies = df['movieId'].unique()

# create mappings from raw IDs to indices
user2index = {u: i for i, u in enumerate(all_users)}
movie2index = {m: i for i, m in enumerate(all_movies)}

# number of unique users and movies
n_users = len(all_users)
n_movies = len(all_movies)
print(f"n_users={n_users}  n_movies={n_movies}")

# encode user and movie IDs in the train and test sets
def encode(df_split):
    mask = df_split['movieId'].isin(movie2index)
    d = df_split[mask]
    u = d['userId'].map(user2index).values
    m = d['movieId'].map(movie2index).values
    r = d['rating'].values.astype(np.float32)
    return u, m, r

u_tr, m_tr, r_tr = encode(train_set)
u_te, m_te, r_te = encode(test_set)
print(f"Train triples: {len(r_tr):,}  |  Test triples: {len(r_te):,}")


n_users=671  n_movies=9066
Train triples: 79,748  |  Test triples: 20,256


### 4. Model
Regularized matrix factorization trained with SGD

**Objective**
$$
\min_{P,Q,b_u,b_i}\sum_{(u,i)\in\mathcal{K}}\!\left(r_{ui} - \mu - b_u - b_i - p_u^\top q_i\right)^2
+ \lambda\!\left(\|p_u\|^2+\|q_i\|^2+b_u^2+b_i^2\right)
$$

| Symbol | Meaning |
|--------|--------|
| $P\in\mathbb{R}^{n_u\times k}$ | User latent matrix |
| $Q\in\mathbb{R}^{n_i\times k}$ | Item latent matrix |
| $b_u, b_i$ | User / item bias |
| $\mu$ | Global mean |
| $\lambda$ | L2 regularization coefficient |


In [6]:
class MatrixFactorization:
    def __init__(self, n_users, n_items, n_factors=20, lr=0.005, reg=0.02, n_epochs=20, seed=42):
        self.n_factors = n_factors # latent factors
        self.lr = lr # learning rate
        self.reg = reg # regularization strength
        self.n_epochs = n_epochs # training epochs

        rng = np.random.default_rng(seed)
        scale = 0.1
        self.P  = rng.normal(0, scale, (n_users, n_factors)).astype(np.float32)   # user factors
        self.Q  = rng.normal(0, scale, (n_items, n_factors)).astype(np.float32)   # item factors
        self.bu = np.zeros(n_users, dtype=np.float32)   # user biases
        self.bi = np.zeros(n_items, dtype=np.float32)   # item biases
        self.mu = 0.0                                    # global mean (set at fit time)


    # prediction -- global mean rating + user bias + item bias + dot product of user and item factors
    def predict(self, u, m):
        return self.mu + self.bu[u] + self.bi[m] + np.sum(self.P[u] * self.Q[m], axis=1)
    
    # training 
    def fit(self, u_tr, m_tr, r_tr, u_te=None, m_te=None, r_te=None, verbose=True):
        # set global mean to the average rating in the training set
        self.mu = r_tr.mean()
        n = len(r_tr)
        idx = np.arange(n)

        # track RMSE history for train and test sets
        self.train_rmse_hist = []
        self.test_rmse_hist  = []

        for epoch in range(1, self.n_epochs + 1):
            np.random.shuffle(idx)      # in-place shuffle for SGD

            # loop over training triples and update parameters
            for i in idx:
                u, m, r = u_tr[i], m_tr[i], r_tr[i]
                err = r - (self.mu + self.bu[u] + self.bi[m] + self.P[u] @ self.Q[m])

                # gradient step
                self.bu[u] += self.lr * (err - self.reg * self.bu[u])
                self.bi[m] += self.lr * (err - self.reg * self.bi[m])
                pu_old      = self.P[u].copy()
                self.P[u]  += self.lr * (err * self.Q[m] - self.reg * self.P[u])
                self.Q[m]  += self.lr * (err * pu_old    - self.reg * self.Q[m])

            # epoch metrics
            tr_rmse = sqrt(mean_squared_error(r_tr, self.predict(u_tr, m_tr)))
            self.train_rmse_hist.append(tr_rmse)

            if u_te is not None:
                te_rmse = sqrt(mean_squared_error(r_te, self.predict(u_te, m_te)))
                self.test_rmse_hist.append(te_rmse)

            if verbose and epoch % 5 == 0:
                msg = f"Epoch {epoch:>3}/{self.n_epochs}  train RMSE={tr_rmse:.4f}"
                if u_te is not None:
                    msg += f"  test RMSE={te_rmse:.4f}"
                print(msg)

        return self


### 5. Train

In [7]:
model = MatrixFactorization(
    n_users=n_users,
    n_items=n_movies,
    n_factors=20,
    lr=0.005,
    reg=0.02,
    n_epochs=30,
)

model.fit(u_tr, m_tr, r_tr, u_te, m_te, r_te)

Epoch   5/30  train RMSE=0.8801  test RMSE=0.9300
Epoch  10/30  train RMSE=0.8496  test RMSE=0.9181
Epoch  15/30  train RMSE=0.8225  test RMSE=0.9139
Epoch  20/30  train RMSE=0.7879  test RMSE=0.9123
Epoch  25/30  train RMSE=0.7450  test RMSE=0.9129
Epoch  30/30  train RMSE=0.7003  test RMSE=0.9141


### 6. Evaluation

In [8]:
def rmse(true, pred): return sqrt(mean_squared_error(true, pred))
def mae(true, pred):  return np.mean(np.abs(true - pred))

train_preds = model.predict(u_tr, m_tr)
test_preds  = model.predict(u_te, m_te)

# Clip predictions to valid rating range
rating_min, rating_max = df['rating'].min(), df['rating'].max()
test_preds_clipped = np.clip(test_preds, rating_min, rating_max)

print("=" * 40)
print(f"Train RMSE : {rmse(r_tr, train_preds):.4f}")
print(f"Test  RMSE : {rmse(r_te, test_preds):.4f}  (raw)")
print(f"Test  RMSE : {rmse(r_te, test_preds_clipped):.4f}  (clipped to [{rating_min},{rating_max}])")
print(f"Test  MAE  : {mae(r_te, test_preds_clipped):.4f}")
print("=" * 40)

Train RMSE : 0.7003
Test  RMSE : 0.9141  (raw)
Test  RMSE : 0.9137  (clipped to [0.5,5.0])
Test  MAE  : 0.6980


### 7. Adjust hyperparams

### 8. Top-N recommendations

In [11]:
# map movieID to their titles
movies = pd.read_csv("../data/movies_metadata.csv", low_memory=False)
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype(int)

links = pd.read_csv("../data/links.csv")
links['tmdbId'] = pd.to_numeric(links['tmdbId'], errors='coerce')
links = links.dropna(subset=['tmdbId'])
links['movieId'] = links['movieId'].astype(int)
links['tmdbId']  = links['tmdbId'].astype(int)

tmdb2title = dict(zip(movies['id'], movies['original_title']))
movielens2title = {
    row.movieId: tmdb2title.get(row.tmdbId, None)
    for row in links.itertuples()
}


def recommend(model, user_raw_id, n=10, seen_movie_ids=None):
    # get user index from raw ID, all movies, and predictions for all movies
    u_idx = user2index[user_raw_id]
    all_m = np.arange(n_movies)
    preds = model.predict(np.full(n_movies, u_idx), all_m)

    # exclude seen movies by setting their predicted scores to -inf
    if seen_movie_ids:
        seen_idx = [movie2index[m] for m in seen_movie_ids if m in movie2index]
        preds[seen_idx] = -np.inf

    # sort predictions and get the top N movie indices, then map back to raw movie IDs and return with predicted scores
    top_idx = np.argsort(preds)[::-1][:n]
    idx2movie = {v: k for k, v in movie2index.items()}

    return [(movielens2title.get(idx2movie[i], "Unknown Title"), round(preds[i], 3)) for i in top_idx]

sample_user = all_users[0]
seen = set(train_set[train_set['userId'] == sample_user]['movieId'])

print(f"Top-10 recommendations for user {sample_user}:")
for rank, (title, score) in enumerate(recommend(model, sample_user, n=10, seen_movie_ids=seen), 1):
    print(f"  {rank:>2}. Movie: {title} | Predicted rating: {score}")



Top-10 recommendations for user 1:
   1. Movie: Happiness | Predicted rating: 3.6989998817443848
   2. Movie: Trois couleurs : Bleu | Predicted rating: 3.6589999198913574
   3. Movie: The Shawshank Redemption | Predicted rating: 3.6570000648498535
   4. Movie: The African Queen | Predicted rating: 3.635999917984009
   5. Movie: On the Waterfront | Predicted rating: 3.556999921798706
   6. Movie: The Godfather: Part II | Predicted rating: 3.5450000762939453
   7. Movie: The Conversation | Predicted rating: 3.5380001068115234
   8. Movie: 12 Angry Men | Predicted rating: 3.5320000648498535
   9. Movie: The Usual Suspects | Predicted rating: 3.5199999809265137
  10. Movie: Amores perros | Predicted rating: 3.5139999389648438


In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from math import sqrt
import random

def mf_score_fn(user_raw_id, candidate_movie_ids, model, user2index, movie2index):
    """Return predicted scores aligned with candidate_movie_ids."""
    u_idx = user2index[user_raw_id]
    m_idx = np.array([movie2index[m] for m in candidate_movie_ids], dtype=np.int32)
    u_vec = np.full(len(candidate_movie_ids), u_idx, dtype=np.int32)
    return model.predict(u_vec, m_idx)

def evaluate_model_mf(
    model,
    train_df,
    test_df,
    user2index,
    movie2index,
    k=10,
    n_candidates=100,
    seed=42
):
    random.seed(seed)
    np.random.seed(seed)

    all_movie_ids = set(movie2index.keys())
    train_seen_by_user = train_df.groupby("userId")["movieId"].apply(set).to_dict()
    test_pos_by_user = test_df.groupby("userId")["movieId"].apply(list).to_dict()

    hits, ndcgs = [], []
    users = [u for u in test_pos_by_user.keys() if u in user2index]

    for user_id in users:
        positives = test_pos_by_user.get(user_id, [])
        if not positives:
            continue

        watched_movie = random.choice(positives)

        # negatives: movies not seen in train/test for this user
        excluded = train_seen_by_user.get(user_id, set()).union(set(positives))
        not_watched = list(all_movie_ids - excluded)
        if len(not_watched) < n_candidates - 1:
            continue

        negative_sample = random.sample(not_watched, n_candidates - 1)
        candidates = [watched_movie] + negative_sample
        random.shuffle(candidates)

        scores = mf_score_fn(user_id, candidates, model, user2index, movie2index)
        ranked_ids = [candidates[i] for i in np.argsort(scores)[::-1]]
        relevance = [int(m == watched_movie) for m in ranked_ids]

        hits.append(int(watched_movie in ranked_ids[:k]))
        ndcg_score = 1 / np.log2(relevance.index(1) + 2) if 1 in relevance else 0.0
        ndcgs.append(ndcg_score)

    return {
        "hit_rate": float(np.mean(hits)) if hits else 0.0,
        "ndcg": float(np.mean(ndcgs)) if ndcgs else 0.0,
        "n_users": len(hits),
        "hit_list": hits,
        "ndcg_list": ndcgs,
    }

# Run evaluation
eval_results = evaluate_model_mf(
    model=model,
    train_df=train_set,
    test_df=test_set,
    user2index=user2index,
    movie2index=movie2index,
    k=10,
    n_candidates=100,
    seed=42,
)

print("MF Ranking Evaluation")
print(f"Users evaluated: {eval_results['n_users']}")
print(f"HitRate@10     : {eval_results['hit_rate']:.4f}")
print(f"NDCG@10        : {eval_results['ndcg']:.4f}")


MF Ranking Evaluation
Users evaluated: 671
HitRate@10     : 0.3040
NDCG@10        : 0.2868
